# Project 3

https://www.kaggle.com/datasets/sonia22222/students-mental-health-assessments/data

## Preprocessing Data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import numpy as np

### Merging the datasets

In [2]:
df_habits = pd.read_csv("df_habits_export.csv")
df_MH = pd.read_csv('df_MH_export.csv')

In [3]:
df_habits.columns = df_habits.columns.str.strip()
df_MH.columns = df_MH.columns.str.strip()

# Rename columns to match
df_habits = df_habits.rename(columns={
    "gender": "Gender",
    "age": "Age",
    "year": "year_of_study",
    "Year": "year_of_study",
    "Year_of_Study": "year_of_study"
})

df_MH = df_MH.rename(columns={
    "gender": "Gender",
    "age": "Age",
    "year": "year_of_study",
    "Year": "year_of_study",
    "Year_of_Study": "year_of_study"
})

#check names after matching
print("df_habits columns:", df_habits.columns.tolist())
print("df_MH columns:", df_MH.columns.tolist())


# Group each dataset
HB_df_1 = (
    df_habits.groupby(["Gender", "Age"])
    .agg({
        "study_hours_per_day": "mean",
        "sleep_hours": "mean",
        "mental_health_rating": "mean"
    })
    .reset_index()
)

SM_df_1 = (
    df_MH.groupby(["Gender", "Age"])
    .agg({
        "CGPA": "mean",
        "Stress_Level": "mean",
        "Depression_Score": "mean",
        "Anxiety_Score": "mean",
        "Financial_Stress": "mean"
    })
    .reset_index()
)

# Merge them together
df_analysis = pd.merge(
    HB_df_1,
    SM_df_1,
    on=["Gender", "Age"],
    how="inner"
)

# View Result
print(df_analysis.head())

print(df_analysis.info())

df_habits columns: ['student_id', 'Age', 'Gender', 'study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality', 'exercise_frequency', 'parental_education_level', 'internet_quality', 'mental_health_rating', 'extracurricular_participation', 'exam_score']
df_MH columns: ['Age', 'Course', 'Gender', 'CGPA', 'Stress_Level', 'Depression_Score', 'Anxiety_Score', 'Sleep_Quality', 'Physical_Activity', 'Diet_Quality', 'Social_Support', 'Relationship_Status', 'Substance_Use', 'Counseling_Service_Use', 'Family_History', 'Chronic_Illness', 'Financial_Stress', 'Extracurricular_Involvement', 'Semester_Credit_Load', 'Residence_Type']
   Gender  Age  study_hours_per_day  sleep_hours  mental_health_rating  \
0  Female   18             3.884746     6.500000              5.627119   
1  Female   19             3.490164     6.527869              5.114754   
2  Female   20             3.644595     6.372973              6.175676   
3  Fe

For this assignment, we used the same merged data file from project 1 and 2. Here we merged selected the columns from each of the data sets that were of interest. Then to create datasets that could be merged together, both were grouped on age and gender and averaged the remaining columns. We then merged the two data sets. It is important to note that to create fair common variables, the grouping and aggregation was required, but by doing that, it significantly reduced the size of the data set.

### Data cleaning and handling of issues: 20 points

In [4]:
# SECTION 1: Data cleaning and handling of issues
# Goal: create a clean modeling dataframe from the required merged dataframe df_analysis.

df_clean = df_analysis.copy()

print(df_clean.dtypes)

numeric_cols = [
    "Age", "study_hours_per_day", "sleep_hours", "mental_health_rating",
    "CGPA", "Stress_Level", "Depression_Score", "Anxiety_Score", "Financial_Stress"
]

outlier_counts = {}
for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outlier_counts[col] = int(((df_clean[col] < lo) | (df_clean[col] > hi)).sum())

print("Cleaned dataframe shape:", df_clean.shape)
print("\nIQR outlier counts:\n", outlier_counts)

Gender                   object
Age                       int64
study_hours_per_day     float64
sleep_hours             float64
mental_health_rating    float64
CGPA                    float64
Stress_Level            float64
Depression_Score        float64
Anxiety_Score           float64
Financial_Stress        float64
dtype: object
Cleaned dataframe shape: (14, 10)

IQR outlier counts:
 {'Age': 0, 'study_hours_per_day': 1, 'sleep_hours': 0, 'mental_health_rating': 0, 'CGPA': 0, 'Stress_Level': 1, 'Depression_Score': 0, 'Anxiety_Score': 0, 'Financial_Stress': 0}


Because of the data processing and manipulation done within the merge, there were no missing values and all the variables of interest were already processed. But, we still preformed a check on data types to ensure everything was correct before moving into the next stage. We also wanted to see if there were any outliers within the data as this could change the way we do the final processing in the pipeline. It was found that there were two found outliers one in study_hours_per_day and 1 in Stress_level. 

### Feature scaling and normalization; one-hot encoding/categorical variable handling: 20 points



In [5]:
y = df_clean["CGPA"].copy()
X = df_clean.drop(columns=["CGPA"]).copy()
num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # used median to preserve outliers instead of mean which would be skewed by them.
    ("scaler", RobustScaler()) # used because dataset is small and has a few outliers but not bad enough to remove. 
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # used most frequent to preserve original categories instead of constant which would create a new category and reduce one-hot encoding effectiveness.
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")) # convert to binary columns 
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

print("Predictor columns:", X.columns.tolist())
print("Numeric features:", num_features)
print("Categorical features:", cat_features)
print("Processed matrix shape:", X_processed.shape)

Predictor columns: ['Gender', 'Age', 'study_hours_per_day', 'sleep_hours', 'mental_health_rating', 'Stress_Level', 'Depression_Score', 'Anxiety_Score', 'Financial_Stress']
Numeric features: ['Age', 'study_hours_per_day', 'sleep_hours', 'mental_health_rating', 'Stress_Level', 'Depression_Score', 'Anxiety_Score', 'Financial_Stress']
Categorical features: ['Gender']
Processed matrix shape: (14, 9)


The pipeline was create to process the numeric and categorical data based on the findings in the previous section. The numeric pipelines used the RobustScaler to handle the outliers within the data without removing them due to the size of our data set. Then the OneHotEnder was used to processes the categorical data (gender) by transforming it all to binary. 


### Dimensionality reduction techniques: 20 points



In [ ]:
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_processed)

print("Original feature count:", X_processed.shape[1])
print("Reduced feature count:", X_pca.shape[1])
print("Explained variance retained:", np.sum(pca.explained_variance_ratio_))

Original feature count: 9
Reduced feature count: 5
Explained variance retained: 0.9158911358062135


To address feature redundancy and potential multicollinearity, we applied PCA for dimensionality reduction. Given the small sample size, we retained 90% of the explained variance, which reduced the feature space from 9 dimensions to 5. This process reduced the risk of model bias by eliminating non-essential information while preserving the core pieces of data. 

### Preprocessing justification and impact analysis: 20 points

To process the data, we first followed the carried over workflow for the previous projects. This included identifying the required columns for our analysis, grouping both data sets by age and gender while averaging all other data points, then merging on common variables (age and gender). To get both of these data sets to a version that allowed for clean merging, the grouping was required, but by doing that it heavily reduced the size of our dataset.

After loading and merging that data, we ensured all of the data types were correct (numeric for all columns but gender as categorical). Then we checked for outliers within the data since the dataset is so small. Any large outliers could heavily impact the performance of the model down the line.

From there I split the data into X (CGPA - the variable to model) and Y (the columns used for predicting). We then created a preprocessing pipeline for both the numeric and categorical data to handle any remaining missing data and the outliers within the numeric data. We used RobustScaler instead of standard scaling because of the small sample size and the outliers. Then we used on-hot encoding on the gender column to transform to binary.

**Impacts:**
Reducing the dimensionality is an important step in the pipeline as it removes variables that are over-represented or highly correlated. By doing this is makes the model more efficiency and removes bias.

## Clustering or Classification Analysis 

###  Clear definition and justification of goals: 10 points

<!-- Regression is appropriate here because CGPA is a continuous numeric outcome.

Goal: predict CGPA using behavioral and mental-health predictors from df_analysis.

Why this choice:
- CGPA is not a class label; forcing bins would discard information.
- Regression keeps the original academic-performance scale and is easier to interpret.
- We will compare multiple regressors and report MAE, RMSE, and R^2. -->

### Implementation of chosen techniques: 20 points

### Parameter optimization and method tuning: 20 points

### Performance evaluation and metric selection: 20 points

### Interpretation and discussion of results: 30 points